# Set the accelerator to GPU T4 x2 when running this code


In [ ]:
import time

NOTEBOOK_START_TIME = time.time()

In [ ]:
DEBUG = False

QUANT = False

if QUANT:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

USE_PAST_KEY = True

In [ ]:
QUANT

In [ ]:
%%time
if QUANT:
    !pip install -U /kaggle/input/accelerate-wheelwhl/accelerate-0.29.1-py3-none-any.whl -qq
    !pip install -U /kaggle/input/bitsandbytes-0-42-0-py3-none-any-whl/bitsandbytes-0.42.0-py3-none-any.whl -qq


import torch
import gc
torch.backends.cuda.enable_mem_efficient_sdp(False)

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    AutoConfig,
    StoppingCriteria,
    set_seed
)

import transformers
print(f"Transformers Version: {transformers.__version__}")
set_seed(42)

In [ ]:
import pandas as pd
from tqdm import tqdm
PRIVATE = True

df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/test.csv')
df.head()

In [ ]:
if len(df) < 5:
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    PRIVATE = False
df.head()

In [ ]:
df.shape

In [ ]:
def naive_parse(answer):
    out = []
    start = False
    end = False
    for l in reversed(list(answer)):
        if l in '0123456789' and not end:
            start = True
            out.append(l)
        else:
            if start:
                end = True
        
    out = reversed(out)
    return ''.join(out)

In [ ]:
import re  # Import the regular expression module
import sys  # Import sys to access system features and operations
import subprocess  # Import subprocess to execute external commands

# Return the last printed content from the output
def return_last_print(output, n):
    lines = output.strip().split('\n')  # Strip surrounding whitespace and split into lines
    if lines:
        return lines[n]  # Return line n
    else:
        return ""  # Return an empty string if there is no content

# Function for processing code
def process_code(code, return_shell_output=False):
    
    # Replace text matching the regular expression
    def repl(match):
        if "real" not in match.group():
            return "{}{}".format(match.group()[:-1], ', real=True)')  # Add real=True when the matched text does not contain "real"
        else:
            return "{}{}".format(match.group()[:-1], ')')  # If "real" is present, append only the closing parenthesis
    
    code = re.sub(r"symbols\([^)]+\)", repl, code)  # Replace symbol definitions using a regular expression

    # When shell output is requested, wrap the code for shell execution
    if return_shell_output:
        code = code.replace('\n', '\n    ')  # Indent each line so it can be nested inside a try block
        # Add exception handling to catch and print execution errors
        code = "\ntry:\n    from sympy import *\n{}\nexcept Exception as e:\n    print(e)\n    print('FAIL')\n".format(code)
    
    # Print the code directly when shell output is not requested
    if not return_shell_output:
        print(code)
    
    # Write the code to 'code.py'
    with open('code.py', 'w') as fout:
        fout.write(code)
    
    # Execute the generated Python code in a subprocess
    batcmd = 'timeout 7 ' + sys.executable + ' code.py'  # Set up the command with a 7-second timeout
    try:
        # Execute the command with subprocess and capture its output
        shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
        # Get the last output line and print the output
        return_value = return_last_print(shell_output, -1)
        print(shell_output)
        
        # When shell output is requested, determine whether execution succeeded
        if return_shell_output:
            if return_value == 'FAIL':  # A return value of "FAIL" indicates an execution failure
                CODE_STATUS = False
                return_value = return_last_print(shell_output, -2)  # Get the error message from the second-to-last line
                if "not defined" in return_value:  # For an undefined-name error, suggest checking formatting and imports
                    return_value += '\nTry checking the formatting and imports'
            else:
                CODE_STATUS = True  # No failure indicates success
            return return_value, CODE_STATUS  # Return the result and status
        
        # Convert a numeric result to an integer and take the remainder
        code_output = round(float(eval(return_value))) % 1000
    except Exception as e:
        print(e, 'shell_output')  # Catch and print the exception
        code_output = -1  # Return -1 on execution failure
    
    # When shell output is requested, determine and return the result status
    if return_shell_output:
        if code_output == -1:
            CODE_STATUS = False
        else:
            CODE_STATUS = True
        return code_output, CODE_STATUS  
    
    return code_output  # Return the result

# Process text output and extract a numeric result
def process_text_output(output):
    result = output  # Assign the output to result
    try:
        # Find output matching \boxed{digits} using a regular expression
        result_output = re.findall(r'\\boxed\{(\d+)\}', result)

        print('BOXED', result_output)
        if not len(result_output):  # If no \boxed match is found, use naive_parse
            result_output = naive_parse(result)
        else:
            result_output = result_output[-1]  # If multiple results are found, use the last one

        print('BOXED FINAL', result_output)
        if not len(result_output):  # Return -1 if no result is available
            result_output = -1
        
        else:
            result_output = round(float(eval(result_output))) % 1000  # Convert the result to a number and take the remainder
    
    except Exception as e:
        print(e)  # Catch and print the exception
        print('ERROR PARSING TEXT')

In [ ]:
torch.cuda.empty_cache()
gc.collect()

# Set private to true to use DeepSeek for local mathematical reasoning

In [ ]:
PRIVATE = True

In [ ]:
import re
import math
import random

from collections import defaultdict

n_repetitions = 22 if PRIVATE else 4
TOTAL_TOKENS = 2048 # if PRIVATE else 512

if PRIVATE:
    TIME_LIMIT = 31500
else:
    TIME_LIMIT = 1

In [ ]:
print(f'n_repetitions;{n_repetitions}')
print(f'TOTAL_TOKENS:{TOTAL_TOKENS}')
print(f'PRIVATE:{PRIVATE}')
print(f'TIME_LIMIT:{TIME_LIMIT}')
print(f'QUANT:{QUANT}')

In [ ]:
import torch
print(torch.cuda.device_count())  # Print the number of available GPUs


In [ ]:
QUANT

In [ ]:
if PRIVATE:

    MODEL_PATH = '/kaggle/input/deepseek-math'#"/root/autodl-tmp/AIMO1/20th/model_data/deepseek-math-7b-rl"#"/kaggle/input/gemma/transformers/7b-it/1"
    DEEP = True

    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.gradient_checkpointing = True

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    device_map = [('model.embed_tokens', 0),
                 ('model.layers.0', 0),
                 ('model.layers.1', 0),
                 ('model.layers.2', 0),
                 ('model.layers.3', 0),
                 ('model.layers.4', 0),
                 ('model.layers.5', 0),
                 ('model.layers.6', 0),
                 ('model.layers.7', 0),
                 ('model.layers.8', 0),
                 ('model.layers.9', 0),
                 ('model.layers.10', 0),
                 ('model.layers.11', 0),
                 ('model.layers.12', 0),
                 ('model.layers.13', 0),
                 ('model.layers.14', 0),
                 ('model.layers.15', 0),
                 ('model.layers.16', 0),
                 ('model.layers.17', 0),
                 ('model.layers.18', 0),
                 ('model.layers.19', 0),
                 ('model.layers.20', 0),
                 ('model.layers.21', 0),
                 ('model.layers.22', 1),
                 ('model.layers.23', 1),
                 ('model.layers.24', 1),
                 ('model.layers.25', 1),
                 ('model.layers.26', 1),
                 ('model.layers.27', 1),
                 ('model.layers.28', 1),
                 ('model.layers.29', 1),
                 ('model.norm', 1),
                 ('lm_head', 1)]

    device_map = {ii:jj for (ii,jj) in device_map}

    if QUANT:
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit = True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            device_map="sequential",
            torch_dtype="auto",
            trust_remote_code=True, 
            quantization_config=quantization_config,
            config=config
        )
    else: # Run this branch when quant==False 
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            device_map=device_map,
            torch_dtype="auto",
            trust_remote_code=True,
            #quantization_config=quantization_config,
            config=config
        )
    # else: # Run this branch when quant==False 
    #     model = AutoModelForCausalLM.from_pretrained(
    #         MODEL_PATH,
    #         device_map='auto',
    #         torch_dtype="auto",
    #         trust_remote_code=True,
    #         #quantization_config=quantization_config,
    #         config=config
    #     )
    pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype='auto',
    device_map=device_map,
)
    from transformers import StoppingCriteriaList

    class StoppingCriteriaSub(StoppingCriteria):
        def __init__(self, stops = [], encounters=1):
            super().__init__()
            self.stops = [stop.to("cuda") for stop in stops]

        def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
            for stop in self.stops:
                last_token = input_ids[0][-len(stop):]
                if torch.all(torch.eq(stop,last_token)):
                    return True
            return False


    stop_words = ["```output", "```python", "```\nOutput" , ")\n```" , "``````output"] #,  
    stop_words_ids = [tokenizer(stop_word, return_tensors='pt', add_special_tokens=False)['input_ids'].squeeze() for stop_word in stop_words]
    stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])
    
    model.dtype, model.hf_device_map

In [ ]:
code = """Below is a math problem you are to solve (positive numerical answer):
\"{}\"
To accomplish this, first determine a sympy-based approach for solving the problem by listing each step to take and what functions need to be called in each step. Be clear so even an idiot can follow your instructions, and remember, your final answer should be positive integer, not an algebraic expression!
Write the entire script covering all the steps (use comments and document it well) and print the result. After solving the problem, output the final numerical answer within \\boxed{}.

Approach:"""


cot = """Below is a math problem you are to solve (positive numerical answer!):
\"{}\"
Analyze this problem and think step by step to come to a solution with programs. After solving the problem, output the final numerical answer within \\boxed{}.\n\n"""

promplt_options = [code,cot]

In [ ]:
promplt_options

In [ ]:
import re
from collections import defaultdict
from collections import Counter

from numpy.random import choice
import numpy as np

tool_instruction = '\n\nPlease integrate natural language reasoning with programs to solve the above problem, and put your final numerical answer within \\boxed{}.\nNote that the intermediary calculations may be real numbers, but the final numercal answer would always be an integer.'


#tool_instruction = " The answer should be given as a non-negative modulo 1000."
#tool_instruction += '\nPlease integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.'

temperature = 0.9
top_p = 1.0

temperature_coding = 0.9
top_p_coding = 1.0
   
total_results = {}
total_answers = {}
best_stats = {}
total_outputs = {}
question_type_counts = {}
starting_counts = (2,3)

In [ ]:
n_repetitions = 1
df = df.head(1)
df

# debug


In [ ]:
for jj in tqdm(range(n_repetitions)):   
    for i in tqdm(range(len(df))):
        TIME_SPENT = time.time() - NOTEBOOK_START_TIME
        if TIME_SPENT>TIME_LIMIT:
            break
        id_ = df['id'].loc[i]
        problem = df['problem'].loc[i]
        print('#'*10)
        print(f"**\n\n\nQUESTION round {jj}, question {i} - elapsed so far: {TIME_SPENT:.0f} secs")
        
        best, best_count = best_stats.get(i,(-1,-1))
        # print(f'best:{best}')
        # print(f'best_count:{best_count}')
        
        if best_count>np.sqrt(jj):
            print("**SKIPPING CAUSE ALREADY FOUND BEST")
            continue
        outputs = total_outputs.get(i,[])
        # print(f'outputs:{outputs}')
        # Initial default values
        text_answers, code_answers = question_type_counts.get(i,starting_counts)# starting_counts is the default when no entry is found
        print(f'text_answers:{text_answers}')
        print(f'code_answers:{code_answers}')
        
        results = total_results.get(i,[])
        answers = total_answers.get(i,[])

        # print(f'results:{results}')
        # print(f'answers:{answers}')
        
        print('**Clear GPU memory in five iterations')
        for _ in range(5):
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(0.2)
        # Generate with the model
        print('###################################Prepare the problem#################################')
        
        try:#
            print('¥¥¥¥¥¥¥¥¥¥¥¥¥Entering try block¥¥¥¥¥¥¥¥¥¥¥¥¥¥')
            ALREADY_GEN = 0
            code_error = None
            code_error_count = 0
            code_output = -1
            #initail_message = problem  + tool_instruction 
            counts = np.array([text_answers,code_answers])
            print(f'**counts:       {counts}')
            #==========================================
            draw = choice(promplt_options, 1,
                          p=counts/counts.sum())
            # draw selects one prompt from promplt_options; p gives each selection probability
            # print(f'Selected prompt template:         {draw}')
            # Insert the problem into the prompt using format
            initail_message = draw[0].format(problem,"{}")   
            
            print(f'############################1--Prompt template with the specific problem########################:')
            prompt = f"User: {initail_message}"
            current_printed = len(prompt)
            # print(f'Unencoded text length after adding User: {current_printed}')
            print(f"{jj}_{prompt}\n")
            model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            input_len = len(model_inputs['input_ids'][0])
            print(f'**Tokenized input_ids length:           {input_len}')
            print('###################################2--Generate the model response#################################')
            generation_output = model.generate(**model_inputs, 
                                               max_new_tokens=TOTAL_TOKENS-ALREADY_GEN,
                                               return_dict_in_generate=USE_PAST_KEY,
                                               do_sample = True,
                                               temperature = temperature,
                                               top_p = top_p,
                                               num_return_sequences=1, stopping_criteria = stopping_criteria)
            print()
            if USE_PAST_KEY:
                output_ids = generation_output.sequences[0]
            else:
                output_ids = generation_output[0]
            # print(f'Check the length of the generated output: {output_ids.shape}')
            
            decoded_output = tokenizer.decode(output_ids, skip_special_tokens=True)
            print('**Decoded model response:')
            print(decoded_output)
            print('**Decoded continuation after removing the original question')
            print(f"{decoded_output[current_printed:]}\n")
            current_printed += len(decoded_output[current_printed:])
            
            print(f'**Current generated response length:{current_printed}')
            cummulative_code = ""
            
            stop_word_cond = False
            print('**Check whether the generated text ends with a stop word')
            print(f'**End of generated text:{decoded_output[-10:]}')
            
            for stop_word in stop_words:
                stop_word_cond = stop_word_cond or (decoded_output[-len(stop_word):]==stop_word)
            print(f'**Stop-word condition satisfied:{stop_word_cond}')  
            while (stop_word_cond) and (ALREADY_GEN<(TOTAL_TOKENS)):
                print('###########3--Continue while the output meets the condition and remains within the length limit############')
                
                
                if (decoded_output[-len("```python"):]=="```python"):
                    temperature_inner=temperature_coding
                    top_p_inner = top_p_coding
                    prompt = decoded_output
                else:
                    temperature_inner=temperature
                    top_p_inner = top_p
                    try:
                        if (decoded_output[-len("``````output"):]=="``````output"):
                            code_text = decoded_output.split('```python')[-1].split("``````")[0]
                        else:
                            code_text = decoded_output.split('```python')[-1].split("```")[0]
                        
                        print('**Start accumulating code')
                        print(f'**Code accumulated so far:{cummulative_code}')
                        print(f'**Newly generated code:{code_text}')
                        
                        cummulative_code+=code_text
                        ######### Write generated code to the code file and execute it to inspect the result
                        code_output, CODE_STATUS = process_code(cummulative_code, return_shell_output=True)
                        print('**CODE RESULTS', code_output)
                        print('**CODE_STATUS', CODE_STATUS)
                        # TOTAL_TOKENS: True indicates successful execution; otherwise the generated program has a problem
                        if code_error==code_output:# Treat a Nane program output as an error
                            
                            code_error_count+=1
                        else:
                            code_error=code_output
                            code_error_count = 0

                        if not CODE_STATUS:# If execution fails, remove the generated program and prepare the prompt for reuse
                            cummulative_code = cummulative_code[:-len(code_text)]# Trim off the newly generated code
                            # Stop the loop if errors persist
                            if code_error_count>=1:
                                print("REPEATED ERRORS")
                                break

                    except Exception as e:
                        print(e)
                        print('ERROR PARSING CODE')
                        code_output = -1

                    if code_output!=-1:
                        if (decoded_output[-len(")\n```"):]==")\n```"):
                            prompt = decoded_output+'```output\n'+str(code_output)+'\n```\n'
                        else:
                            prompt = decoded_output+'\n'+str(code_output)+'\n```\n'
                    else:
                        prompt = decoded_output
                        cummulative_code=""

                
                print('%'*200)
                print('Leaving the loop here')
                print('########################4--Start the second model generation call########################')
                
                print('**prompt:',prompt)
                model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                ALREADY_GEN =  len(model_inputs['input_ids'][0])-input_len

                if USE_PAST_KEY:
                    old_values = generation_output.past_key_values
                else:
                    old_values = None

                generation_output = model.generate(**model_inputs, 
                                                   max_new_tokens=TOTAL_TOKENS-ALREADY_GEN, 
                                                   return_dict_in_generate=USE_PAST_KEY,
                                                   past_key_values=old_values,
                                                   do_sample = True,
                                                   temperature = temperature_inner,
                                                   top_p = top_p_inner,
                                                   num_return_sequences=1, stopping_criteria = stopping_criteria)

                if USE_PAST_KEY:
                    output_ids = generation_output.sequences[0]
                else:
                    output_ids = generation_output[0]
                decoded_output = tokenizer.decode(output_ids, skip_special_tokens=True)

                print('**Print generated code following the proposed approach')
                print(f"**\nINTERMEDIATE OUT :\n{decoded_output[current_printed:]}\n")
                current_printed+=len(decoded_output[current_printed:])
                
                stop_word_cond = False
                for stop_word in stop_words:
                    stop_word_cond = stop_word_cond or (decoded_output[-len(stop_word):]==stop_word)

                # Check the updated stop_word_cond here to determine whether the while loop should stop
                print()
            ####################################################### End of the while loop #######################################################



            
            print()
            print()
            print()
            
            print('**Code generation has ended; proceed to postprocessing without further model-generated Python')
            if USE_PAST_KEY:
                output_ids = generation_output.sequences[0]
            else:
                output_ids = generation_output[0]

            raw_output = tokenizer.decode(output_ids[input_len:], skip_special_tokens=True)
            #print(f"\n\nOutput :\n{raw_output}\n")                            
            result_output = process_text_output(raw_output)
            
            try:
                code_output = round(float(eval(code_output))) % 1000
            except Exception as e:
                print(e,'final_eval')
                code_output = -1

        except Exception as e:
            print(e,"5")
            result_output, code_output = -1, -1

        if code_output!=-1:
            outputs.append(code_output)
            code_answers+=1

        if result_output!=-1:
            outputs.append(result_output)
            text_answers+=1

        if len(outputs) > 0:
            occurances = Counter(outputs).most_common()
            print(occurances)
            if occurances[0][1] > best_count:
                print("GOOD ANSWER UPDATED!")
                best = occurances[0][0]
                best_count = occurances[0][1]
            if occurances[0][1] > 5:
                print("ANSWER FOUND!")
                break

        results.append(result_output)
        answers.append(code_output)
        
        best_stats[i] = (best, best_count) 
        question_type_counts[i] = (text_answers, code_answers)
        total_outputs[i] = outputs
        
        total_results[i] = results
        total_answers[i] = answers

        print("code_answers",code_answers-starting_counts[1],"text_answers",text_answers-starting_counts[0])
        if DEBUG:
            break
            
        print('Code execution complete')

In [ ]:
1

In [ ]:
import numpy as np
if PRIVATE:
    for ii in range(len(df)):
        a = total_answers[ii]
        b = total_answers[ii]
        a = np.array(a)
        b = np.array(b)
        print(a,b)
        a[a < 0] = b[a < 0]

        pred = Counter(a.tolist()).most_common(2)
        print(pred)

In [ ]:
if PRIVATE:
    df['answer'] = [best_stats[ii][0] for ii in range(len(df))]
else:
    df['answer'] = 2

In [ ]:
df.head()

In [ ]:
if not PRIVATE:
    print(1)
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    if PRIVATE:
        df['model_answer'] = [best_stats[ii][0] for ii in range(len(df))]
        df['match'] = df.answer == df.model_answer
        print(f'{df.match.sum()} matches in {len(df)} examples')

In [ ]:
df

In [ ]:
with open('code.py', 'w') as fout:
    fout.write("print('done')")

batcmd = 'timeout 7 ' + sys.executable + ' code.py'
try:
    shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
    print(shell_output)
except:
    pass

# Test code below

# Test converting a code string into executable code

In [ ]:
OK_CODE = '''
from sympy import symbols, Eq, solve, sqrt, simplify

def distance_squared_sum():
    x = symbols('x')
    k, l = symbols('k l', positive=True, real=True)

    # Set the equations equal to each other
    equation = Eq(k*x**2 - 2*k*x + l, 4)
    
    # Solve for x
    solutions = solve(equation, x)
    A, B = solutions

    # Calculate the distances from A and B to the origin
    dA = sqrt(A**2 + (k*A**2 - 2*k*A + l)**2)
    dB = sqrt(B**2 + (k*B**2 - 2*k*B + l)**2)

    # Simplify the expressions
    dA = simplify(dA)
    dB = simplify(dB)

    # Calculate the sum of the squares of the distances
    sum_of_squares = dA**2 + dB**2

    return sum_of_squares.subs({k: 1, l: 2})

result = distance_squared_sum()
print(result)
'''


In [ ]:
# Function for processing code
def process_code_DeBUG(code, return_shell_output=False):
    
    # Replace text matching the regular expression
    print(f'Code before repl processing:{code}')
    def repl(match):
        if "real" not in match.group():
            return "{}{}".format(match.group()[:-1], ', real=True)')  # Add real=True when the matched text does not contain "real"
        else:
            return "{}{}".format(match.group()[:-1], ')')  # If "real" is present, append only the closing parenthesis
    
    code = re.sub(r"symbols\([^)]+\)", repl, code)  # Replace symbol definitions using a regular expression
    print(f'Code after processing:{code}')

    # When shell output is requested, wrap the code for shell execution
    if return_shell_output:
        code = code.replace('\n', '\n    ')  # Indent each line so it can be nested inside a try block
        # Add exception handling to catch and print execution errors
        code = "\ntry:\n    from sympy import *\n{}\nexcept Exception as e:\n    print(e)\n    print('FAIL')\n".format(code)
    
    # Print the code directly when shell output is not requested
    if not return_shell_output:
        print(code)
    
    # Write the code to 'code.py'
    with open('code.py', 'w') as fout:
        fout.write(code)
    
    # Execute the generated Python code in a subprocess
    batcmd = 'timeout 7 ' + sys.executable + ' code.py'  # Set up the command with a 7-second timeout
    try:
        # Execute the command with subprocess and capture its output
        shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
        # Get the last output line and print the output
        return_value = return_last_print(shell_output, -1)
        print(shell_output)
        
        # When shell output is requested, determine whether execution succeeded
        if return_shell_output:
            if return_value == 'FAIL':  # A return value of "FAIL" indicates an execution failure
                CODE_STATUS = False
                return_value = return_last_print(shell_output, -2)  # Get the error message from the second-to-last line
                if "not defined" in return_value:  # For an undefined-name error, suggest checking formatting and imports
                    return_value += '\nTry checking the formatting and imports'
            else:
                CODE_STATUS = True  # No failure indicates success
            return return_value, CODE_STATUS  # Return the result and status
        
        # Convert a numeric result to an integer and take the remainder
        code_output = round(float(eval(return_value))) % 1000
    except Exception as e:
        print(e, 'shell_output')  # Catch and print the exception
        code_output = -1  # Return -1 on execution failure
    
    # When shell output is requested, determine and return the result status
    if return_shell_output:
        if code_output == -1:
            CODE_STATUS = False
        else:
            CODE_STATUS = True
        return code_output, CODE_STATUS  
    
    return code_output  # Return the result


In [ ]:
code_output_11, CODE_STATUS_11 = process_code_DeBUG(OK_CODE, return_shell_output=True)
# code_output: the answer produced by the program
# CODE_STATUS: whether the program executed successfully

In [ ]:
code_output_11

In [ ]:
CODE_STATUS_11